In [1]:
import numpy as np
import pandas as pd

In [2]:

movies = pd.read_pickle(r"..\\data\processed\movies.pkl")
movies.sample(7)

,movie_id,title,tag,title_normalized
1959,859,Dangerous Liaisons,drama romance lovesickness sexuality cheating ...,dangerous liaisons
82,119450,Dawn of the Planet of the Apes,sciencefiction action drama thriller leader co...,dawn of the planet of the apes
4652,20520,The Motel,drama comedy thirteen-year-old ernest chin liv...,the motel
553,4349,The Kingdom,thriller action drama assassination assassin t...,the kingdom
2966,24137,An Ideal Husband,drama comedy romance sir robert chiltern is a ...,an ideal husband
2815,67660,Think Like a Man,comedy romance advice love relationship ensemb...,think like a man
1257,12704,Ishtar,action adventure comedy wilderness sahara beau...,ishtar


### Stemming

In [3]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [4]:
ps.stem('loved')

'love'

In [5]:
def stem(text):
    y = []

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

In [6]:
stem("Hello I am Akshat Sonkar")

'hello i am akshat sonkar'

In [7]:
stem("This is am unique way of writing and reading")

'thi is am uniqu way of write and read'

In [8]:
movies['tag'].apply(stem)

0       action adventur fantasi sciencefict culturecla...
1       adventur fantasi action ocean drugabus exotici...
2       action adventur crime spi basedonnovel secreta...
3       action crime drama thriller dccomic crimefight...
4       action adventur sciencefict basedonnovel mar m...
                              ...                        
4804    action crime thriller unitedstates–mexicobarri...
4805    comedi romanc a newlyw couple' honeymoon is up...
4806    comedi drama romanc tvmovi date loveatfirstsig...
4807    when ambiti new york attorney sam is sent to s...
4808    documentari obsess camcord crush dreamgirl eve...
Name: tag, Length: 4806, dtype: str

In [9]:
movies['tag'] = movies['tag'].apply(stem)

### Vectorization of Movies dataset's tags

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')

In [11]:
cv.fit_transform(movies['tag']).toarray().shape


(4806, 5000)

In [12]:
vectors = cv.fit_transform(movies['tag']).toarray()

In [13]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0], shape=(5000,))

In [14]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      shape=(5000,), dtype=object)

### Finding similarity using Cosine

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
cosine_similarity(vectors).shape

(4806, 4806)

In [17]:
similarity = cosine_similarity(vectors)

In [18]:
import pickle
from pathlib import Path

similarity_file_path = Path.cwd().parent / "data" / "processed" / "similarity.pkl"
with similarity_file_path.open("wb") as file:
    pickle.dump(similarity, file)

print(f"Similarity matrix saved to: {similarity_file_path}")

Similarity matrix saved to: c:\Coding\Project Black Knight\data\processed\similarity.pkl


In [19]:
similarity

array([[1.        , 0.08346223, 0.0860309 , ..., 0.04499213, 0.        ,
        0.        ],
       [0.08346223, 1.        , 0.06063391, ..., 0.02378257, 0.        ,
        0.02615329],
       [0.0860309 , 0.06063391, 1.        , ..., 0.02451452, 0.        ,
        0.        ],
       ...,
       [0.04499213, 0.02378257, 0.02451452, ..., 1.        , 0.03962144,
        0.04229549],
       [0.        , 0.        , 0.        , ..., 0.03962144, 1.        ,
        0.08714204],
       [0.        , 0.02615329, 0.        , ..., 0.04229549, 0.08714204,
        1.        ]], shape=(4806, 4806))

In [20]:
similarity[0]

array([1.        , 0.08346223, 0.0860309 , ..., 0.04499213, 0.        ,
       0.        ], shape=(4806,))

### Recommendation Function

In [21]:
movies.head()

,movie_id,title,tag,title_normalized
0,19995,Avatar,action adventur fantasi sciencefict culturecla...,avatar
1,285,Pirates of the Caribbean: At World's End,adventur fantasi action ocean drugabus exotici...,pirates of the caribbean at worlds end
2,206647,Spectre,action adventur crime spi basedonnovel secreta...,spectre
3,49026,The Dark Knight Rises,action crime drama thriller dccomic crimefight...,the dark knight rises
4,49529,John Carter,action adventur sciencefict basedonnovel mar m...,john carter


In [22]:
similarity

array([[1.        , 0.08346223, 0.0860309 , ..., 0.04499213, 0.        ,
        0.        ],
       [0.08346223, 1.        , 0.06063391, ..., 0.02378257, 0.        ,
        0.02615329],
       [0.0860309 , 0.06063391, 1.        , ..., 0.02451452, 0.        ,
        0.        ],
       ...,
       [0.04499213, 0.02378257, 0.02451452, ..., 1.        , 0.03962144,
        0.04229549],
       [0.        , 0.        , 0.        , ..., 0.03962144, 1.        ,
        0.08714204],
       [0.        , 0.02615329, 0.        , ..., 0.04229549, 0.08714204,
        1.        ]], shape=(4806, 4806))

In [23]:
import sys
from pathlib import Path

src_file_path = Path.cwd().parent
sys.path.insert(0,str(src_file_path))
from src.text_normalizer import Text_Normalization

In [24]:
norm = Text_Normalization()

def normalizeinput (text):
    return norm.normalize(text)

In [25]:
from difflib import get_close_matches

def recommend(movie):

    movienorm = normalizeinput(movie)

    if movienorm not in movies['title_normalized'].values:
        close_matches = get_close_matches(
            movienorm,
            movies['title_normalized'].tolist(),
            n=1,
            cutoff=0.6
        )

        if not close_matches:
            print(f"{movie} not found.")
            return None

        movienorm = close_matches[0]
        matched_title = movies.loc[
            movies['title_normalized'] == movienorm, 'title'
        ].iloc[0]
        print(f"Using closest match: {matched_title}")
    
    movie_index = movies[movies['title_normalized'] == movienorm].index[0]
    distance = similarity[movie_index]
    movies_list = sorted(list(enumerate(distance)), reverse=True, key=lambda x:x[1])[1:6]      #This part is explained below but can only be used for movie names

    result = []

    for i in movies_list:
        result.append({'title': movies.iloc[i[0]].title,
                       'similarity_score': i[1]})
        #print(movies.iloc[i[0]].title)
    return pd.DataFrame(result)

In [26]:
recommend('Dhurandhar')

Using closest match: DragonHeart


,title,similarity_score
0,Dragonslayer,0.300541
1,Maleficent,0.264255
2,How to Train Your Dragon,0.242789
3,Underworld: Rise of the Lycans,0.240823
4,Your Highness,0.239681


In [27]:
recommend('avatr')

Using closest match: Avatar


,title,similarity_score
0,Aliens vs Predator: Requiem,0.286770
1,Aliens,0.269014
2,Falcon Rising,0.260513
3,Independence Day,0.255609
4,Titan A.E.,0.250387


In [28]:
movies[movies['title'] == 'Batman Begins'].index[0]

np.int64(119)

In [29]:
recommend("GodFather")

Using closest match: The Godfather


,title,similarity_score
0,Desert Dancer,0.451378
1,Step Up,0.385122
2,Take the Lead,0.378378
3,Footloose,0.372822
4,Tango,0.348025


In [30]:
recommend("batman begins")

,title,similarity_score
0,The Dark Knight,0.402181
1,Batman,0.354342
2,Batman,0.334077
3,The Dark Knight Rises,0.317744
4,10th & Wolf,0.312010


In [31]:
similarity[0]

array([1.        , 0.08346223, 0.0860309 , ..., 0.04499213, 0.        ,
       0.        ], shape=(4806,))

In [32]:
enumerate(similarity[0])

In [33]:
list(enumerate(similarity[0]))      #converted into tupple which gives distance of 0 with 1,2,3.. and so on

[(0, np.float64(1.0000000000000002)),
 (1, np.float64(0.08346223261119858)),
 (2, np.float64(0.08603090020146065)),
 (3, np.float64(0.0734718358370645)),
 (4, np.float64(0.1892994097121204)),
 (5, np.float64(0.10838874619051501)),
 (6, np.float64(0.04024218182927669)),
 (7, np.float64(0.14673479641335554)),
 (8, np.float64(0.05923488777590923)),
 (9, np.float64(0.0978231976089037)),
 (10, np.float64(0.10259783520851541)),
 (11, np.float64(0.09464970485606021)),
 (12, np.float64(0.09037128496931669)),
 (13, np.float64(0.04499212706658476)),
 (14, np.float64(0.12988108336653278)),
 (15, np.float64(0.06282808624375433)),
 (16, np.float64(0.07894736842105264)),
 (17, np.float64(0.13977653617040256)),
 (18, np.float64(0.09493290614465533)),
 (19, np.float64(0.0830812984794528)),
 (20, np.float64(0.058038100008800934)),
 (21, np.float64(0.10968169942141635)),
 (22, np.float64(0.0662266178532522)),
 (23, np.float64(0.08740748201220976)),
 (24, np.float64(0.0533380747062665)),
 (25, np.float64

In [34]:
sorted(list(enumerate(similarity[0])))      #sorted based on first value we need based on second value

[(0, np.float64(1.0000000000000002)),
 (1, np.float64(0.08346223261119858)),
 (2, np.float64(0.08603090020146065)),
 (3, np.float64(0.0734718358370645)),
 (4, np.float64(0.1892994097121204)),
 (5, np.float64(0.10838874619051501)),
 (6, np.float64(0.04024218182927669)),
 (7, np.float64(0.14673479641335554)),
 (8, np.float64(0.05923488777590923)),
 (9, np.float64(0.0978231976089037)),
 (10, np.float64(0.10259783520851541)),
 (11, np.float64(0.09464970485606021)),
 (12, np.float64(0.09037128496931669)),
 (13, np.float64(0.04499212706658476)),
 (14, np.float64(0.12988108336653278)),
 (15, np.float64(0.06282808624375433)),
 (16, np.float64(0.07894736842105264)),
 (17, np.float64(0.13977653617040256)),
 (18, np.float64(0.09493290614465533)),
 (19, np.float64(0.0830812984794528)),
 (20, np.float64(0.058038100008800934)),
 (21, np.float64(0.10968169942141635)),
 (22, np.float64(0.0662266178532522)),
 (23, np.float64(0.08740748201220976)),
 (24, np.float64(0.0533380747062665)),
 (25, np.float64

In [35]:
sorted(list(enumerate(similarity[0])), reverse=True, key=lambda x:x[1])

[(0, np.float64(1.0000000000000002)),
 (1214, np.float64(0.28676966733820225)),
 (2405, np.float64(0.26901379342448517)),
 (3728, np.float64(0.2605130246476754)),
 (507, np.float64(0.255608593705383)),
 (539, np.float64(0.25038669783359574)),
 (582, np.float64(0.24511108480187255)),
 (1202, np.float64(0.24455799402225925)),
 (1192, np.float64(0.2367785320221084)),
 (61, np.float64(0.23179316248638276)),
 (778, np.float64(0.23174488732966073)),
 (4046, np.float64(0.2278389747471728)),
 (1916, np.float64(0.2252817784447915)),
 (2782, np.float64(0.21853668936906193)),
 (172, np.float64(0.21239769762143662)),
 (972, np.float64(0.2108663315950723)),
 (322, np.float64(0.2105263157894737)),
 (2329, np.float64(0.20443988269091456)),
 (3606, np.float64(0.20437977982832192)),
 (260, np.float64(0.20395079136182276)),
 (151, np.float64(0.2029530274475215)),
 (4190, np.float64(0.2029530274475215)),
 (1440, np.float64(0.20277677641345318)),
 (74, np.float64(0.2024645717996314)),
 (1087, np.float64(0

In [36]:
sorted(list(enumerate(similarity[0])), reverse=True, key=lambda x:x[1])[1:6]

[(1214, np.float64(0.28676966733820225)),
 (2405, np.float64(0.26901379342448517)),
 (3728, np.float64(0.2605130246476754)),
 (507, np.float64(0.255608593705383)),
 (539, np.float64(0.25038669783359574))]

In [37]:
recommend("Batman Begins")

,title,similarity_score
0,The Dark Knight,0.402181
1,Batman,0.354342
2,Batman,0.334077
3,The Dark Knight Rises,0.317744
4,10th & Wolf,0.312010


In [38]:
movies.iloc[1216].title

'Autumn in New York'

In [39]:
recommend('avatar')

,title,similarity_score
0,Aliens vs Predator: Requiem,0.286770
1,Aliens,0.269014
2,Falcon Rising,0.260513
3,Independence Day,0.255609
4,Titan A.E.,0.250387


In [40]:
import pickle

with open("../data/models/similarity.pkl", "wb") as f:
    pickle.dump(similarity, f)

with open("../data/models/vectorizer.pkl", "wb") as f:
    pickle.dump(cv, f)